# Signalling-Bots on Colab

1. Runtime -> Change runtime type -> GPU.
2. Add a GitHub token as a Colab secret called `GITHUB_TOKEN` (key icon in the left sidebar, turn on notebook access).
3. Run the cells in order. All seeds run together in one process.
4. The last cells commit the results and log and push them to GitHub.


In [ ]:
!nvidia-smi -L
import jax
print("jax", jax.__version__, jax.devices())

In [ ]:
REPO = "https://github.com/migarak2908/Signalling-Bots"
BRANCH = "main"

import os
os.chdir("/content")
if not os.path.exists("Signalling-Bots"):
    !git clone -b {BRANCH} {REPO}
os.chdir("/content/Signalling-Bots")
!git pull
!git log --oneline -1

In [ ]:
COLOURS = 3  # Stage A: 2 colours, seeds 0-9. Stage B: 3 colours, seeds 0-19.
SEEDS = " ".join(str(s) for s in range(20))
GENERATIONS = 5000

LOG = f"results/logs/{COLOURS}col_g{GENERATIONS}_colab.log"
!mkdir -p results/logs
!python -u -m bots.evolve --colours {COLOURS} --seeds {SEEDS} --generations {GENERATIONS} 2>&1 | tee {LOG}


In [ ]:
!python -m bots.summarize {COLOURS}col


In [ ]:
from pathlib import Path
from IPython.display import Image, display

for run in sorted(Path("results", f"{COLOURS}col").glob(f"s*_g{GENERATIONS}")):
    print(run.name)
    for plot in ("fitness.png", "trajectories.png"):
        display(Image(str(run / plot), width=900))


In [ ]:
# Commit all results and logs, and push them to GitHub.
import subprocess
from google.colab import userdata

TOKEN_SECRET = "GITHUB_TOKEN"  # name of the Colab secret holding your GitHub token
token = userdata.get(TOKEN_SECRET)

!git config user.name "migarak2908"
!git config user.email "migarak2908@gmail.com"
!git add results/
!git commit -q -m "Add {COLOURS}col results from Colab (seeds {SEEDS}, {GENERATIONS} generations)"

url = f"https://{token}@github.com/migarak2908/Signalling-Bots.git"
for cmd in (["git", "pull", "--rebase", "-q", url, BRANCH], ["git", "push", url, f"HEAD:{BRANCH}"]):
    out = subprocess.run(cmd, capture_output=True, text=True)
    print((out.stdout + out.stderr).replace(token, "***"))
    if out.returncode != 0:
        break

In [ ]:
# Optional: also copy the results to Google Drive.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/Signalling-Bots/results
!cp -r results/{COLOURS}col /content/drive/MyDrive/Signalling-Bots/results/